# 🛰️ Rondônia SR Study — Phase 2: Training + Evaluation

**This notebook requires GPU** (T4 or P100 — select in Notebook Settings → Accelerator)

**Before running:**
- Accelerator: **GPU T4 x1** (Settings panel, right side)
- Internet: **ON**
- Add these datasets via '+ Add Data':
  - `rondonia-sr-code` (your uploaded code dataset)
  - `rondonia-sr-patches` (output of Phase 1 notebook)

**Training schedule** (one model at a time to stay within 30 GPU-hours/week):

| Model | Epochs | Est. Time (T4) |
|---|---|---|
| SRCNN | 150 | ~1.5h |
| EDSR | 300 | ~4h |
| RCAN | 300 | ~5h |
| SRGAN | 200 | ~4h |
| ESRGAN | 200 | ~4h |
| SwinIR | 400 | ~8h |
| HAT | 300 | ~6h |

⚠️ **Total is ~35h** — use the `MODEL_TO_TRAIN` cell to run one model per session.

---
## Step 1 — Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════
# EDIT THESE — choose which model to train this session
# Options: "srcnn" | "edsr" | "rcan" | "srgan" | "esrgan" | "swinir" | "hat"
MODEL_TO_TRAIN = "edsr"
SCALING        = "smart"    # "standard" or "smart"

# Run full evaluation after training? (adds ~30min)
RUN_EVAL       = True

# Run downstream segmentation task after eval? (adds ~30min)
RUN_DOWNSTREAM = True
# ════════════════════════════════════════════════════════════════

import os, sys, shutil
from pathlib import Path

WORK_DIR     = Path("/kaggle/working")
CODE_DIR     = Path("/kaggle/input/rondonia-sr-code")
PATCHES_DIR  = Path("/kaggle/input/rondonia-sr-patches")

print(f"Model      : {MODEL_TO_TRAIN.upper()}")
print(f"Scaling    : {SCALING}")
print(f"Code dir   : {CODE_DIR.exists()}")
print(f"Patches dir: {PATCHES_DIR.exists()}")

import torch
print(f"CUDA       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Step 2 — Install Dependencies

In [ ]:
%%bash
pip install -q \
    rasterio>=1.3.9 \
    scikit-image>=0.22.0 \
    scipy>=1.11.0 \
    lpips>=0.1.4 \
    scikit-learn>=1.3.0 \
    torchmetrics>=1.2.0 \
    timm>=0.9.12 \
    pyyaml>=6.0.1 \
    tqdm>=4.66.0 \
    pandas>=2.0.0 \
    tabulate>=0.9.0 \
    matplotlib>=3.8.0
echo "All packages installed."

---
## Step 3 — Set Up Code + Data

In [ ]:
import shutil, sys, zipfile
from pathlib import Path

WORK_DIR    = Path("/kaggle/working")
CODE_DIR    = Path("/kaggle/input/rondonia-sr-code")
PATCHES_INPUT = Path("/kaggle/input/rondonia-sr-patches")

# ── Copy code to working dir ──────────────────────────────────────────────
if not CODE_DIR.exists():
    raise RuntimeError("rondonia-sr-code dataset not found! Add it via '+ Add Data'.")

for item in CODE_DIR.iterdir():
    dest = WORK_DIR / item.name
    if item.is_dir():
        if dest.exists(): shutil.rmtree(dest)
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

sys.path.insert(0, str(WORK_DIR))
print("✓ Code files copied")

# ── Extract patches ───────────────────────────────────────────────────────
if not PATCHES_INPUT.exists():
    raise RuntimeError("rondonia-sr-patches dataset not found! Run Phase 1 first.")

zip_candidates = list(PATCHES_INPUT.glob("*.zip"))
aligned_dir = WORK_DIR / "data" / "aligned"

if zip_candidates:
    print(f"Extracting {zip_candidates[0].name}...")
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        zf.extractall(WORK_DIR)
    print("✓ Patches extracted")
elif (PATCHES_INPUT / "data" / "aligned").exists():
    # Already unzipped in dataset
    shutil.copytree(PATCHES_INPUT / "data" / "aligned", aligned_dir, dirs_exist_ok=True)
    print("✓ Patches copied from dataset")
else:
    raise RuntimeError("Cannot find patches in dataset. Expected a .zip or data/aligned/ directory.")

# Count
for split in ["train", "val", "test"]:
    n = len(list((aligned_dir / split).glob("*.npz"))) if (aligned_dir / split).exists() else 0
    print(f"  {split:5s}: {n} patches")

# ── Create output dirs ────────────────────────────────────────────────────
for d in [WORK_DIR / "checkpoints", WORK_DIR / "results"]:
    d.mkdir(parents=True, exist_ok=True)
print("✓ Directories ready")

---
## Step 4 — Patch config.yaml for Kaggle

In [ ]:
import yaml
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
cfg_path = WORK_DIR / "config.yaml"

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Paths
cfg["paths"]["landsat_dir"]  = str(WORK_DIR / "data" / "raw" / "landsat")
cfg["paths"]["sentinel_dir"] = str(WORK_DIR / "data" / "raw" / "sentinel")
cfg["paths"]["aligned_dir"]  = str(WORK_DIR / "data" / "aligned")
cfg["paths"]["checkpoints"]  = str(WORK_DIR / "checkpoints")
cfg["paths"]["results"]      = str(WORK_DIR / "results")

# Kaggle GPU settings (from config.yaml cloud section)
cloud = cfg.get("cloud", {})
# Use Kaggle profile if present
kaggle_overrides = cloud.get("kaggle", {})
if MODEL_TO_TRAIN in kaggle_overrides.get("models_to_run", [MODEL_TO_TRAIN]):
    cfg["training"]["batch_size"]   = kaggle_overrides.get("batch_size", 4)
    cfg["training"]["accum_steps"]  = kaggle_overrides.get("accum_steps", 8)
    cfg["training"]["amp_dtype"]    = kaggle_overrides.get("amp_dtype", "bfloat16")
    cfg["training"]["num_workers"]  = kaggle_overrides.get("num_workers", 2)
    cfg["training"]["pin_memory"]   = True
    print(f"Using Kaggle profile: BS={cfg['training']['batch_size']}, "
          f"accum={cfg['training']['accum_steps']}, "
          f"eff_BS={cfg['training']['batch_size'] * cfg['training']['accum_steps']}")

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print("✓ config.yaml patched")

---
## Step 5 — Train Selected Model

> ⚠️ This is the long-running step. Keep the tab open.  
> Kaggle notebooks timeout after ~9h — make sure `MODEL_TO_TRAIN` can finish in time.

In [ ]:
import subprocess, sys, time
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

cmd = [
    sys.executable, str(WORK_DIR / "train.py"),
    "--model",   MODEL_TO_TRAIN,
    "--scaling", SCALING,
    "--config",  str(WORK_DIR / "config.yaml"),
]

# Check if checkpoint exists to resume
ckpt = WORK_DIR / "checkpoints" / f"{MODEL_TO_TRAIN}_{SCALING}" / "latest.pt"
if ckpt.exists():
    cmd += ["--resume", str(ckpt)]
    print(f"Resuming from: {ckpt}")

print(f"Starting: {' '.join(cmd[2:])}")
print("=" * 60)

t0 = time.time()
result = subprocess.run(cmd, cwd=str(WORK_DIR), capture_output=False)
elapsed = (time.time() - t0) / 3600

if result.returncode != 0:
    print(f"\n⚠️  Training exited with errors (code {result.returncode})")
else:
    print(f"\n✓ Training complete ({elapsed:.2f}h)")
    best = WORK_DIR / "checkpoints" / f"{MODEL_TO_TRAIN}_{SCALING}" / "best.pt"
    if best.exists():
        size_mb = best.stat().st_size / 1e6
        print(f"  Best checkpoint: {best.name} ({size_mb:.0f} MB)")

---
## Step 6 — Evaluation (PSNR / SSIM / SAM / ERGAS / LPIPS)

In [ ]:
if not RUN_EVAL:
    print("Skipping evaluation (RUN_EVAL=False)")
else:
    import subprocess, sys
    from pathlib import Path

    WORK_DIR = Path("/kaggle/working")

    result = subprocess.run(
        [
            sys.executable, str(WORK_DIR / "evaluate.py"),
            "--model",      MODEL_TO_TRAIN,
            "--scaling",    SCALING,
            "--test-split", "test",
            "--config",     str(WORK_DIR / "config.yaml"),
        ],
        cwd=str(WORK_DIR),
        capture_output=False,
    )

    if result.returncode == 0:
        import pandas as pd
        csv = WORK_DIR / "results" / "evaluation_results.csv"
        if csv.exists():
            df = pd.read_csv(csv)
            row = df[(df["model"] == MODEL_TO_TRAIN) & (df["scaling"] == SCALING)]
            if not row.empty:
                print("\n📊 Results:")
                print(row[["model","scaling","psnr","ssim","sam_deg","ergas","lpips"]].to_string(index=False))

---
## Step 7 — Downstream Segmentation Task
Trains a lightweight U-Net on SR outputs to assess forest/deforestation/road detection quality.

In [ ]:
if not RUN_DOWNSTREAM:
    print("Skipping downstream task (RUN_DOWNSTREAM=False)")
else:
    import subprocess, sys
    from pathlib import Path

    WORK_DIR = Path("/kaggle/working")

    result = subprocess.run(
        [
            sys.executable,
            str(WORK_DIR / "downstream" / "train_downstream.py"),
            "--sr-model", MODEL_TO_TRAIN,
            "--scaling",  SCALING,
            "--config",   str(WORK_DIR / "config.yaml"),
        ],
        cwd=str(WORK_DIR),
        capture_output=False,
    )

    if result.returncode == 0:
        import pandas as pd
        csv = WORK_DIR / "results" / "downstream_results.csv"
        if csv.exists():
            df = pd.read_csv(csv)
            print("\n📊 Downstream Results:")
            print(df.to_string(index=False))

---
## Step 8 — Save Outputs
Zips checkpoints + results for download.

In [ ]:
import zipfile
from pathlib import Path
from tqdm import tqdm

WORK_DIR = Path("/kaggle/working")
out_zip  = WORK_DIR / f"rondonia_sr_{MODEL_TO_TRAIN}_{SCALING}_output.zip"

to_zip = [
    *list((WORK_DIR / "checkpoints").rglob("*.pt")),
    *list((WORK_DIR / "results").rglob("*.csv")),
    *list((WORK_DIR / "results").rglob("*.md")),
    *list((WORK_DIR / "results").rglob("*.png")),
]

print(f"Packing {len(to_zip)} files...")
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in tqdm(to_zip):
        zf.write(f, f.relative_to(WORK_DIR))

size_mb = out_zip.stat().st_size / 1e6
print(f"\n✓ Saved: {out_zip.name}  ({size_mb:.0f} MB)")
print("")
print("To download:")
print("  Kaggle → Output tab → click the zip file → Download")
print("")
print(f"Repeat Phase 2 for other models by changing MODEL_TO_TRAIN at the top.")

---
## 📋 Training Schedule — Run in This Order

Each session: change `MODEL_TO_TRAIN` at the top → **Run All**

| Session | Model | Scaling | Est. GPU hours |
|---|---|---|---|
| 1 | `srcnn` | `standard` + `smart` | ~3h |
| 2 | `edsr` | `standard` + `smart` | ~8h |
| 3 | `rcan` | `standard` + `smart` | ~10h |
| 4 | `srgan` | `standard` + `smart` | ~8h |
| 5 | `esrgan` | `standard` + `smart` | ~8h |
| 6 | `swinir` | `smart` only | ~8h |
| 7 | `hat` | `smart` only | ~6h |

**Total: ~51 GPU hours** across ~2 weeks of free Kaggle sessions.

After all models are trained, run `evaluate.py` without `--model` to generate the full comparison table.